# Planes auto-editables: lazo abierto vs. replanificación
## Avance 2 — versión simplificada

**Inteligencia Artificial: Representación y Solución de Problemas** · Universidad del Rosario · Prof. Nicolás Avilán Vargas

> ¿Bajo qué condiciones del entorno un agente que reescribe su propio plan supera a uno con plan estático, y a partir de qué costo de reescritura la ventaja se invierte?

En el Avance 1 se propuso modelar en MESA un agente recolector con seis arquitecturas (de reflejo simple a lazo cerrado con gobernanza) y un oráculo de replanificación con competencia $p$ y costo $c$, así como un conjunto amplio de parámetros con los cuales desplazar la frontera entre plan estático y auto-edición. Este cuaderno implementa una versión reducida de ese modelo con el objetivo de aislar el núcleo de la pregunta, es decir el plano $(\lambda, c)$ donde se busca el punto de cruce, por lo que cualquier resultado posterior se podrá leer como una desviación respecto a este caso base y no como el efecto mezclado de seis parámetros a la vez.

## ¿En qué consiste la simplificación?

| Se conserva | Se quita o se fija |
|---|---|
| Retícula de von Neumann con obstáculos y base fija ($15\times15$, 15 % de obstáculos, 6 recursos) | Observabilidad parcial: se fija $r=\infty$ |
| Recursos que se mueven con probabilidad $\lambda$ por tick | Oráculo estocástico con fallos plausibles: se fija $p=1$ (A* exacto) |
| Deliberar ocupa $c$ ticks mientras el mundo sigue cambiando | Políticas basada en modelo, reinicio en frío y gobernada (presupuesto $B$ de ediciones) |
| El plan se calcula sobre la foto del mundo al *iniciar* la deliberación | Representación del plan como programa editable (aquí el plan es una lista de celdas) |
| Políticas reflejo, lazo abierto y lazo cerrado | Capacidad $>1$: se fija en 1, sin trabajo en vuelo ni `carga_perdida` |
| Utilidad neta $= 10\cdot\text{entregas} - \text{ticks deliberando}$ | Métricas `ceguera` y `latencia_deteccion`, barrido sobre $\kappa$ |

La simplificación consiste en conservar únicamente el compromiso central (deliberar cuesta tiempo, y ese tiempo vale más cuanto más rápido cambia el mundo) e idealizar todo lo demás, de forma que ambos planificadores ven el tablero completo y su replanificador nunca se equivoca, con el único costo de quedarse quietos $c$ ticks mientras los recursos siguen moviéndose. Cabe mencionar que la elección es deliberadamente favorable a la replanificación, pues observabilidad total más oráculo perfecto es el mejor caso posible para el lazo cerrado, lo cual convierte al resultado en una cota superior: Si replanificar pierde aquí en alguna región del plano $(\lambda, c)$, perderá también cuando se reintroduzcan la creencia obsoleta y los fallos del oráculo.

La regla de costo es la misma para los dos planificadores, de modo que toda replanificación de reparación (la que dispara un fallo o una divergencia) cuesta $c$ ticks, mientras que los planes de rutina (el plan inicial, el regreso a la base tras recoger y el nuevo encargo tras entregar) son gratuitos, pues equivalen al plan que se entrega de antemano al agente, de modo que solo se pagan las reparaciones. La regla es justa porque el experimento compara dos criterios sobre *cuándo* replanificar (al detectar la divergencia o al fallar) y no una replanificación pagada contra una gratuita, por lo que ambos agentes usan el mismo replanificador al mismo precio y solo difieren en el momento en que lo invocan.

- **Reflejo (`REFLEX`):** Va con avidez al recurso más cercano sin plan ni memoria, con un 15 % de movimientos aleatorios ($\varepsilon=0.15$) para escapar de mínimos locales.
- **Lazo abierto (`OPEN_LOOP`):** Calcula una ruta A* y la ejecuta a ciegas, de modo que solo descubre el fallo al llegar a una celda vacía, momento en el cual delibera $c$ ticks quieto y aplica un plan calculado con el mundo tal como estaba al empezar.
- **Lazo cerrado (`CLOSED_LOOP`):** Monitorea en cada tick y, ante una divergencia, delibera $c$ ticks quieto con el mismo mecanismo, de forma que repara antes de caminar hacia la celda equivocada.

En términos PEAS el entorno queda con la utilidad neta en 300 ticks como medida de rendimiento, la retícula con recursos móviles como entorno, mover, recoger, entregar y deliberar como actuadores, y visión total del tablero como sensores (totalmente observable, estocástico, dinámico, secuencial y discreto). Por último, se descartan y se vuelven a generar los mapas en los que la componente conexa de la base cubre menos de la mitad de las celdas, pues con la base encerrada en una bolsa pequeña la corrida compararía mapas y no políticas.

Las figuras de la corrida y del barrido se muestran como reproductores interactivos (play, pausa y deslizador), y donde no corre JavaScript basta con fijar `MODO = "estatico"` en la celda siguiente para obtener las figuras fijas.

In [ ]:
import importlib.util

if importlib.util.find_spec("mesa") is None:  # p. ej. en Colab
    get_ipython().run_line_magic("pip", "install -q mesa")

from heapq import heappop, heappush
from itertools import count

from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import mesa
import numpy as np
import pandas as pd

MODO = "animacion"  # "estatico" para figuras fijas donde no corre JavaScript
plt.rcParams["animation.embed_limit"] = 25  # MB permitidos por reproductor embebido


def reproductor(fig, dibujar, n_cuadros, intervalo=250):
    """Reproductor HTML (play, pausa y deslizador) que funciona sin kernel."""
    anim = FuncAnimation(fig, dibujar, frames=n_cuadros, interval=intervalo, blit=False)
    html = anim.to_jshtml()
    plt.close(fig)  # la figura vive dentro del reproductor, no como imagen suelta
    return HTML(html)


print("Mesa", mesa.__version__)

Mesa 3.5.1


## Implementación

In [2]:
TAMANO = 15                # lado de la retícula (sin toro)
DENSIDAD_OBSTACULOS = 0.15
N_RECURSOS = 6             # K: recursos simultáneos
EPSILON = 0.15             # probabilidad de paso aleatorio del agente reflejo
W_ENTREGA = 10.0           # w: valor de cada entrega
KAPPA = 1.0                # κ: precio de cada tick de deliberación
PASOS = 300                # horizonte de cada corrida
BASE = (0, 0)
MOVIMIENTOS = ((1, 0), (0, 1), (-1, 0), (0, -1))  # von Neumann, orden fijo
POLITICAS = ("REFLEX", "OPEN_LOOP", "CLOSED_LOOP")

In [3]:
def manhattan(a, b):
    """Distancia de Manhattan: heurística admisible y consistente en von Neumann."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def vecinos(pos, transitable):
    """Celdas contiguas transitables, siempre en el mismo orden."""
    x, y = pos
    return [(x + dx, y + dy) for dx, dy in MOVIMIENTOS if transitable((x + dx, y + dy))]


def astar(inicio, meta, transitable):
    """Ruta óptima de ``inicio`` a ``meta`` (ambos incluidos); lista vacía si no hay.

    Los empates en ``f`` se rompen por orden de inserción (contador), de modo que la
    misma consulta devuelve siempre la misma ruta.
    """
    orden = count()
    frontera = [(manhattan(inicio, meta), next(orden), inicio)]
    g = {inicio: 0}
    padre = {inicio: None}
    while frontera:
        _, _, actual = heappop(frontera)
        if actual == meta:
            ruta = []
            while actual is not None:
                ruta.append(actual)
                actual = padre[actual]
            return ruta[::-1]
        for q in vecinos(actual, transitable):
            ng = g[actual] + 1
            if ng < g.get(q, float("inf")):
                g[q], padre[q] = ng, actual
                heappush(frontera, (ng + manhattan(q, meta), next(orden), q))
    return []

In [4]:
class Recurso(mesa.Agent):
    """Recurso que deriva: con probabilidad λ por tick salta a una celda vecina."""

    def __init__(self, model, pos):
        super().__init__(model)
        self.pos = pos

    def step(self):
        if self.random.random() < self.model.lam:
            opciones = vecinos(self.pos, self.model.transitable)
            if opciones:
                self.pos = self.random.choice(opciones)

In [5]:
class Recolector(mesa.Agent):
    """Agente recolector con capacidad 1 y una de tres políticas.

    - ``REFLEX``: sin plan; paso ávido hacia la meta, con escape aleatorio ε.
    - ``OPEN_LOOP``: plan A* fijo; solo descubre la deriva al llegar y fallar.
    - ``CLOSED_LOOP``: vigila su objetivo y replanifica al detectar una divergencia.

    Toda replanificación de reparación (tras un fallo o una divergencia) cuesta c ticks
    a ambas políticas; los planes de rutina (inicial, hacia la base, tras entregar) son
    gratuitos.
    """

    def __init__(self, model, politica):
        super().__init__(model)
        self.politica = politica
        self.pos = BASE
        self.cargando = False
        self.plan = None          # lista de celdas por recorrer; None = sin plan
        self.objetivo = None      # Recurso perseguido
        self.creencia = None      # dónde se cree que está el objetivo
        self.pendiente = None     # plan en deliberación (calculado al iniciarla)
        self.restantes = 0        # ticks de deliberación que faltan
        self.fallo = False        # el plan anterior terminó en una celda vacía
        self.entregas = self.ticks_deliberando = 0
        self.n_replan = self.tareas_fallidas = 0
        self.trayectoria = [BASE]

    def step(self):
        if self.restantes > 0:
            self._deliberar()
        elif self.politica == "REFLEX":
            self._paso_reflejo()
        else:
            self._paso_planificado()
        self.trayectoria.append(self.pos)

    # --- REFLEX
    def _paso_reflejo(self):
        self._al_llegar()
        meta = BASE if self.cargando else self.model.mas_cercano(self.pos).pos
        opciones = vecinos(self.pos, self.model.transitable)
        if self.random.random() < EPSILON:
            self.pos = self.random.choice(opciones)
        else:
            d = manhattan(self.pos, meta)
            mejores = [q for q in opciones if manhattan(q, meta) < d]
            self.pos = self.random.choice(mejores or opciones)
        self._al_llegar()

    def _al_llegar(self):
        """Recogida y entrega automáticas del agente reflejo."""
        if self.cargando and self.pos == BASE:
            self._entregar()
        elif not self.cargando:
            aqui = [r for r in self.model.recursos() if r.pos == self.pos]
            if aqui:
                self._recoger(aqui[0])

    # --- OPEN_LOOP y CLOSED_LOOP
    def _paso_planificado(self):
        divergencia = (
            self.politica == "CLOSED_LOOP"
            and self.plan is not None
            and self.objetivo is not None
            and self.objetivo.pos != self.creencia
        )
        if divergencia or self.fallo:            # reparación: cuesta c ticks
            self.fallo = False
            self.n_replan += 1
            self.pendiente = self._planificar()  # instantánea del mundo al empezar
            if self.model.c > 0:
                self.restantes = self.model.c
                self._deliberar()
                return
            self._instalar()                     # c = 0: replanifica y actúa
        if self.plan is None:
            self.pendiente = self._planificar()
            self._instalar()
        if self.plan:
            self.pos = self.plan.pop(0)
        if not self.plan:
            self._fin_de_plan()

    def _planificar(self):
        """(plan, objetivo, creencia) con A* sobre las posiciones verdaderas."""
        if self.cargando:
            return astar(self.pos, BASE, self.model.transitable)[1:], None, None
        r = self.model.mas_cercano(self.pos)
        return astar(self.pos, r.pos, self.model.transitable)[1:], r, r.pos

    def _instalar(self):
        self.plan, self.objetivo, self.creencia = self.pendiente
        self.pendiente = None

    def _deliberar(self):
        """Un tick quieto pensando; al completar c ticks se instala el plan."""
        self.ticks_deliberando += 1
        self.restantes -= 1
        if self.restantes == 0:
            self._instalar()

    def _fin_de_plan(self):
        if self.cargando and self.pos == BASE:
            self._entregar()
        elif not self.cargando and self.objetivo.pos == self.pos:
            self._recoger(self.objetivo)
        else:
            self.tareas_fallidas += 1   # el recurso ya no estaba donde se creía
            self.fallo = True           # se repara (pagando c) en el tick siguiente
        self.plan = self.objetivo = self.creencia = None

    # --- acciones comunes
    def _recoger(self, recurso):
        recurso.remove()
        self.cargando = True

    def _entregar(self):
        self.cargando = False
        self.entregas += 1
        self.model.reponer()

In [6]:
REPORTEROS = {
    "entregas": lambda m: m.recolector.entregas,
    "ticks_deliberando": lambda m: m.recolector.ticks_deliberando,
    "n_replan": lambda m: m.recolector.n_replan,
    "tareas_fallidas": lambda m: m.recolector.tareas_fallidas,
    "utilidad_neta": lambda m: (
        W_ENTREGA * m.recolector.entregas - KAPPA * m.recolector.ticks_deliberando
    ),
}


class Mundo(mesa.Model):
    """Retícula con obstáculos, K recursos que derivan y un recolector."""

    def __init__(self, politica="CLOSED_LOOP", lam=0.1, c=2, seed=None):
        super().__init__(rng=seed)
        self.lam, self.c = lam, c
        self.libres = []
        # Se descartan los mapas en que los obstáculos encierran la base en una bolsa
        # pequeña: con ellos la corrida no compara políticas sino mapas.
        while len(self.libres) < TAMANO * TAMANO // 2:
            self.obstaculos = {
                (x, y)
                for x in range(TAMANO)
                for y in range(TAMANO)
                if (x, y) != BASE and self.random.random() < DENSIDAD_OBSTACULOS
            }
            self.libres = self._componente_de_la_base()
        for _ in range(N_RECURSOS):
            self.reponer()
        self.recolector = Recolector(self, politica)
        self.datacollector = mesa.DataCollector(model_reporters=REPORTEROS)
        self.datacollector.collect(self)

    def transitable(self, q):
        return 0 <= q[0] < TAMANO and 0 <= q[1] < TAMANO and q not in self.obstaculos

    def _componente_de_la_base(self):
        """Celdas alcanzables desde la base (BFS), ordenadas para reproducibilidad."""
        vistos, frontera = {BASE}, [BASE]
        while frontera:
            for q in vecinos(frontera.pop(), self.transitable):
                if q not in vistos:
                    vistos.add(q)
                    frontera.append(q)
        return sorted(vistos)

    def recursos(self):
        return list(self.agents_by_type.get(Recurso, []))

    def mas_cercano(self, pos):
        """Recurso más cercano por Manhattan; empates por orden de creación."""
        return min(self.recursos(), key=lambda r: (manhattan(pos, r.pos), r.unique_id))

    def reponer(self):
        """Crear un recurso en una celda libre al azar de la componente de la base."""
        candidatas = [q for q in self.libres if q != BASE]
        if candidatas:
            Recurso(self, self.random.choice(candidatas))

    def step(self):
        self.recolector.step()                       # el agente actúa...
        self.agents_by_type[Recurso].shuffle_do("step")  # ...y el mundo deriva
        self.datacollector.collect(self)


def correr(politica, lam, c, seed, pasos=PASOS):
    """Ejecuta una corrida; devuelve (métricas finales, modelo)."""
    modelo = Mundo(politica, lam, c, seed)
    for _ in range(pasos):
        modelo.step()
    final = modelo.datacollector.get_model_vars_dataframe().iloc[-1].to_dict()
    return {"politica": politica, "lam": lam, "c": c, "seed": seed, **final}, modelo

Antes del barrido conviene ver una sola corrida con $\lambda=0.25$ y $c=2$, en la que las tres políticas parten del mismo mapa y de los mismos recursos iniciales, de forma que el reproductor muestra lado a lado los primeros 120 ticks de cada una (los recursos reales en naranja, la creencia del agente sobre su objetivo como un rombo hueco y el plan restante como una línea delgada) junto con las entregas acumuladas debajo, cuyo cursor avanza con el tick. Se puede reproducir la animación completa o bien arrastrar el deslizador hasta el tick en que el lazo cerrado empieza a deliberar (el agente se vuelve rojo y el subtítulo indica `deliberando k/c`), lo cual permite notar que el lazo abierto camina hasta una celda que ya quedó vacía antes de detenerse a deliberar, mientras que el lazo cerrado se detiene apenas el rombo deja de coincidir con el recurso, así como si alguna de las dos pausas se compensa con entregas más frecuentes (la tabla bajo el reproductor resume la corrida completa de 300 ticks).

In [7]:
LAM, C, SEMILLA = 0.25, 2, 7
PASOS_ANIM = 120           # ticks del reproductor (la tabla resume los 300)
CADA = 1                   # quedarse con uno de cada CADA ticks para aligerar el cuaderno
COLORES = {"REFLEX": "tab:gray", "OPEN_LOOP": "tab:red", "CLOSED_LOOP": "tab:blue"}
C_OBSTACULO, C_BASE, C_RECURSO = "#9e9e9e", "tab:green", "tab:orange"
C_AGENTE, C_DELIBERA = "tab:blue", "tab:red"


def instantanea(modelo, delibero):
    """Lo que se dibuja de un tick: mundo real, creencia y plan del recolector."""
    a = modelo.recolector
    return {
        "agente": a.pos,
        "recursos": [r.pos for r in modelo.recursos()],
        "creencia": a.creencia,
        "plan": [a.pos] + list(a.plan or []),
        "deliberando": (modelo.c - a.restantes, modelo.c) if delibero else None,
        "cargando": a.cargando,
        "entregas": a.entregas,
    }


def trazar(politica, pasos=PASOS_ANIM):
    """Corre el modelo guardando una instantánea por tick (solo lee, no altera nada)."""
    modelo = Mundo(politica, LAM, C, SEMILLA)
    cuadros = [instantanea(modelo, False)]
    for _ in range(pasos):
        antes = modelo.recolector.ticks_deliberando
        modelo.step()
        cuadros.append(instantanea(modelo, modelo.recolector.ticks_deliberando > antes))
    return modelo, cuadros


trazas = {p: trazar(p) for p in POLITICAS}
indices = list(range(0, PASOS_ANIM + 1, CADA))

fig = plt.figure(figsize=(11.5, 5.8), dpi=75)
gs = fig.add_gridspec(2, 3, height_ratios=[3.2, 1.25], hspace=0.38, wspace=0.08,
                      left=0.04, right=0.98, top=0.93, bottom=0.08)
paneles = {}
for k, p in enumerate(POLITICAS):
    ax = fig.add_subplot(gs[0, k])
    mod = trazas[p][0]
    malla = np.zeros((TAMANO, TAMANO))
    for x, y in mod.obstaculos:
        malla[y, x] = 1
    ax.imshow(np.ma.masked_equal(malla, 0), cmap=ListedColormap([C_OBSTACULO]), origin="lower")
    ax.scatter(*BASE, s=150, marker="s", color=C_BASE, zorder=2)
    ax.set(xlim=(-0.5, TAMANO - 0.5), ylim=(-0.5, TAMANO - 0.5), xticks=[], yticks=[])
    paneles[p] = {
        "ax": ax,
        "plan": ax.plot([], [], lw=1, color=C_AGENTE, alpha=0.7, zorder=3)[0],
        "recursos": ax.scatter([], [], s=55, color=C_RECURSO, zorder=4),
        "creencia": ax.scatter([], [], s=130, marker="D", facecolors="none",
                               edgecolors="0.15", lw=1.3, zorder=5),
        "agente": ax.scatter([], [], s=120, color=C_AGENTE, zorder=6),
    }

ax_e = fig.add_subplot(gs[1, :2])
lineas = {p: ax_e.step([], [], where="post", color=COLORES[p], label=p)[0] for p in POLITICAS}
cursor = ax_e.axvline(0, color="0.4", lw=1, ls="--")
techo = max(cs[-1]["entregas"] for _, cs in trazas.values())
ax_e.set(xlim=(0, PASOS_ANIM), ylim=(0, techo + 1), xlabel="tick", ylabel="entregas acumuladas")
ax_e.legend(fontsize=8, loc="upper left")
ax_e.grid(alpha=0.3)

ax_l = fig.add_subplot(gs[1, 2])
ax_l.axis("off")
simbolos = [
    plt.Line2D([], [], marker="o", ls="", color=C_AGENTE, ms=9, label="agente"),
    plt.Line2D([], [], marker="o", ls="", color=C_DELIBERA, ms=9, label="agente deliberando"),
    plt.Line2D([], [], marker="o", ls="", color=C_AGENTE, mec=C_RECURSO, mew=2, ms=9,
               label="agente con carga"),
    plt.Line2D([], [], marker="o", ls="", color=C_RECURSO, ms=7, label="recurso (real)"),
    plt.Line2D([], [], marker="D", ls="", mfc="none", mec="0.15", ms=8,
               label="creencia sobre el objetivo"),
    plt.Line2D([], [], color=C_AGENTE, lw=1, alpha=0.7, label="plan restante"),
    plt.Line2D([], [], marker="s", ls="", color=C_BASE, ms=9, label="base"),
    plt.Line2D([], [], marker="s", ls="", color=C_OBSTACULO, ms=9, label="obstáculo"),
]
ax_l.legend(handles=simbolos, loc="center", ncol=2, fontsize=8, frameon=False)


def _xy(celdas):
    return np.array(celdas, dtype=float).reshape(-1, 2)


def dibujar(j):
    t = indices[j]
    for p, art in paneles.items():
        f = trazas[p][1][t]
        art["recursos"].set_offsets(_xy(f["recursos"]))
        art["creencia"].set_offsets(_xy([f["creencia"]] if f["creencia"] else []))
        plan = _xy(f["plan"]) if len(f["plan"]) > 1 else _xy([])
        art["plan"].set_data(plan[:, 0], plan[:, 1])
        art["agente"].set_offsets(_xy([f["agente"]]))
        art["agente"].set_facecolor(C_DELIBERA if f["deliberando"] else C_AGENTE)
        art["agente"].set_edgecolor(C_RECURSO if f["cargando"] else "none")
        art["agente"].set_linewidth(2.2 if f["cargando"] else 0)
        if f["deliberando"]:
            estado, color = "deliberando {}/{}".format(*f["deliberando"]), C_DELIBERA
        elif p == "REFLEX":
            estado, color = "sin plan (reflejo)", "0.35"
        else:
            estado, color = "ejecutando el plan", "0.35"
        art["ax"].set_title(f"{p}  ·  tick {t}  ·  entregas {f['entregas']}", fontsize=10)
        art["ax"].set_xlabel(estado, color=color, fontsize=10)
    for p, linea in lineas.items():
        serie = [c["entregas"] for c in trazas[p][1][: t + 1]]
        linea.set_data(range(len(serie)), serie)
    cursor.set_xdata([t, t])
    return []


if MODO == "animacion":
    display(reproductor(fig, dibujar, len(indices), intervalo=250))
else:
    dibujar(len(indices) - 1)
    plt.show()

pd.DataFrame([correr(p, LAM, C, SEMILLA)[0] for p in POLITICAS]).set_index("politica")

,lam,c,seed,entregas,ticks_deliberando,n_replan,tareas_fallidas,utilidad_neta
politica,,,,,,,,
REFLEX,0.25,2,7,7.0,0.0,0.0,0.0,70.0
OPEN_LOOP,0.25,2,7,11.0,42.0,21.0,21.0,68.0
CLOSED_LOOP,0.25,2,7,8.0,100.0,50.0,0.0,-20.0


Con $\lambda=0$ el lazo abierto y el lazo cerrado deben coincidir *exactamente*, semilla a semilla, pues sin dinamismo los recursos nunca abandonan su celda, el plan inicial nunca caduca, el lazo abierto nunca falla y el monitoreo nunca encuentra una divergencia que dispare la deliberación. Cualquier diferencia indicaría un error de implementación y no un hallazgo, por lo que la celda siguiente lo verifica con aserciones sobre 8 semillas y $c\in\{0,2,10\}$, exigiendo entregas idénticas y cero reparaciones y ticks de deliberación en ambas políticas antes de continuar.

In [8]:
filas = []
for seed in range(8):
    for c in (0, 2, 10):
        abierto, _ = correr("OPEN_LOOP", 0.0, c, seed)
        cerrado, _ = correr("CLOSED_LOOP", 0.0, c, seed)
        assert abierto["entregas"] == cerrado["entregas"], (seed, c)
        assert abierto["n_replan"] == cerrado["n_replan"] == 0
        assert abierto["ticks_deliberando"] == cerrado["ticks_deliberando"] == 0
        filas.append({"seed": seed, "c": c, "entregas OPEN_LOOP": abierto["entregas"],
                      "entregas CLOSED_LOOP": cerrado["entregas"]})
print("Control superado: con λ = 0 ambas políticas coinciden exactamente.")
pd.DataFrame(filas).pivot(index="seed", columns="c")

Control superado: con λ = 0 ambas políticas coinciden exactamente.


entregas OPEN_LOOP             entregas CLOSED_LOOP            
c                    0     2     10                   0     2     10
seed                                                                
0                  14.0  14.0  14.0                 14.0  14.0  14.0
1                  13.0  13.0  13.0                 13.0  13.0  13.0
2                  12.0  12.0  12.0                 12.0  12.0  12.0
3                  13.0  13.0  13.0                 13.0  13.0  13.0
4                  11.0  11.0  11.0                 11.0  11.0  11.0
5                  14.0  14.0  14.0                 14.0  14.0  14.0
6                  12.0  12.0  12.0                 12.0  12.0  12.0
7                  14.0  14.0  14.0                 14.0  14.0  14.0

El barrido recorre $\lambda \in \{0, 0.1, 0.25, 0.5\}$ y $c \in \{0, 1, 2, 5, 10\}$ para las tres políticas con 10 semillas (600 corridas de 300 ticks), donde se incluye $c=1$ (el menor costo positivo en ticks enteros) porque con la rejilla $\{0, 2, 5, 10\}$ el cruce quedaba entre $c=0$ y $c=2$ sin poder distinguir si un solo tick bastaba para invertir la ventaja. El reproductor muestra la utilidad neta en función de $\lambda$ para un valor de $c$ a la vez, con el eje vertical fijo para que los cuadros se puedan comparar entre sí, de forma que al arrastrar el deslizador de $c=0$ hacia $c=10$ se ve cómo la curva del lazo cerrado cruza por debajo de la del lazo abierto (es decir, el punto donde la ventaja se invierte), mientras que la banda corresponde al error estándar entre semillas (con solo 10 réplicas, vale la pena anotarlo, un cruce dentro de las bandas no debe leerse como concluyente). La última tabla reporta, para los dos planificadores, el número medio de reparaciones (`n_replan`) y de ticks deliberando, que es donde se lee el mecanismo detrás de las curvas.

In [9]:
LAMBDAS = (0.0, 0.1, 0.25, 0.5)
COSTOS = (0, 1, 2, 5, 10)
SEMILLAS = range(10)

df = pd.DataFrame([
    correr(p, lam, c, s)[0]
    for lam in LAMBDAS for c in COSTOS for p in POLITICAS for s in SEMILLAS
])

resumen = (df.groupby(["c", "politica", "lam"])["utilidad_neta"]
             .agg(media="mean", ee="sem").reset_index())


def curvas(ax, c):
    """Utilidad neta contra λ para un costo c: una línea por política, banda ± e.e."""
    for p in POLITICAS:
        d = resumen[(resumen.c == c) & (resumen.politica == p)]
        ax.plot(d.lam, d.media, marker="o", label=p, color=COLORES[p])
        ax.fill_between(d.lam, d.media - d.ee, d.media + d.ee, alpha=0.2, color=COLORES[p])
    ax.set(title=f"c = {c}", xlabel="λ (dinamismo)", xticks=LAMBDAS)
    ax.grid(alpha=0.3)


if MODO == "animacion":
    # Eje vertical fijo para todos los cuadros, de modo que se puedan comparar entre sí.
    piso = (resumen.media - resumen.ee).min()
    tope = (resumen.media + resumen.ee).max()
    margen = 0.05 * (tope - piso)
    fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=80)

    def dibujar_c(k):
        ax.clear()
        curvas(ax, COSTOS[k])
        ax.set(ylim=(piso - margen, tope + margen), ylabel="utilidad neta (media ± e.e.)")
        ax.legend(loc="lower left")
        return []

    display(reproductor(fig, dibujar_c, len(COSTOS), intervalo=1200))
else:
    fig, ejes = plt.subplots(1, len(COSTOS), figsize=(18, 4), sharey=True)
    for ax, c in zip(ejes, COSTOS):
        curvas(ax, c)
    ejes[0].set_ylabel("utilidad neta (media ± e.e.)")
    ejes[0].legend()
    plt.tight_layout()
    plt.show()

df.pivot_table(index="lam", columns=["c", "politica"], values="utilidad_neta").round(1)

c                 0                            1                    \
politica CLOSED_LOOP OPEN_LOOP REFLEX CLOSED_LOOP OPEN_LOOP REFLEX   
lam                                                                  
0.00           125.0     125.0   60.0       125.0     125.0   60.0   
0.10           122.0     123.0   54.0       104.0     118.1   54.0   
0.25           125.0     118.0   59.0        68.5     106.2   59.0   
0.50           128.0     114.0   86.0       -11.3      79.8   86.0   

c                 2                            5                    \
politica CLOSED_LOOP OPEN_LOOP REFLEX CLOSED_LOOP OPEN_LOOP REFLEX   
lam                                                                  
0.00           125.0     125.0   60.0       125.0     125.0   60.0   
0.10            80.0     103.4   54.0        21.0      59.0   54.0   
0.25             2.4      66.8   59.0      -145.0     -20.6   59.0   
0.50          -121.4      27.2   86.0      -244.4    -100.2   86.0   

c                 10                   
politica CLOSED_LOOP OPEN_LOOP REFLEX  
lam                                    
0.00           125.0     125.0   60.0  
0.10          -116.6      10.5   54.0  
0.25          -216.0    -125.4   59.0  
0.50          -281.3    -156.6   86.0

In [10]:
(df[df.politica != "REFLEX"]
   .pivot_table(index=["c", "lam"], columns="politica",
                values=["n_replan", "ticks_deliberando"])
   .round(1))

n_replan           ticks_deliberando          
politica CLOSED_LOOP OPEN_LOOP       CLOSED_LOOP OPEN_LOOP
c  lam                                                    
0  0.00          0.0       0.0               0.0       0.0
   0.10         13.2       7.0               0.0       0.0
   0.25         35.4      11.0               0.0       0.0
   0.50         72.7      16.7               0.0       0.0
1  0.00          0.0       0.0               0.0       0.0
   0.10         14.0       7.9              14.0       7.9
   0.25         41.5      13.8              41.5      13.8
   0.50        101.3      22.2             101.3      22.2
2  0.00          0.0       0.0               0.0       0.0
   0.10         16.0       6.8              32.0      13.6
   0.25         43.3      17.1              86.6      34.2
   0.50         91.3      29.9             182.4      59.8
5  0.00          0.0       0.0               0.0       0.0
   0.10         15.0       9.2              75.0      46.0
   0.25         39.3      19.4             194.0      95.6
   0.50         53.8      29.9             267.4     148.2
10 0.00          0.0       0.0               0.0       0.0
   0.10         17.3       8.2             171.6      80.5
   0.25         24.9      17.7             247.0     173.4
   0.50         29.2      19.2             289.3     189.6

## ¿Qué muestran los resultados?

Primeramente, con $\lambda=0$ ambos planificadores obtienen 125 de utilidad en todas las posiciones del deslizador, lo cual confirma el control, mientras que con $c=0$ el lazo cerrado supera al abierto con $\lambda=0.25$ (125 frente a 118) y con $\lambda=0.5$ (128 frente a 114), con un empate práctico en $\lambda=0.1$ (122 frente a 123). Progresivamente, al cobrar un solo tick por reparación la relación se invierte en todo el rango, de forma que con $c=1$ el lazo abierto gana en $\lambda=0.1$ (118.1 frente a 104.0), en $\lambda=0.25$ (106.2 frente a 68.5) y en $\lambda=0.5$ (79.8 frente a −11.3), por lo que **en este caso idealizado el cruce se ubica entre $c=0$ y $c=1$, es decir que la ventaja del lazo cerrado no resiste ni el menor costo de deliberación**. El mecanismo está en la tabla de reparaciones, pues el lazo cerrado repara ante cada deriva de su objetivo mientras que el abierto solo lo hace al fallar (varias derivas seguidas cuentan como un único fallo), lo cual con $\lambda=0.5$ y $c=1$ da 101.3 reparaciones frente a 22.2, y con $c=10$ deja al lazo cerrado 289.3 de los 300 ticks deliberando. El reflejo, que nunca paga deliberación, se mantiene entre 54 y 86 y supera a ambos planificadores cuando $\lambda c$ crece (86 frente a 27.2 y −121.4 con $c=2$ y $\lambda=0.5$). Finalmente, cabe mencionar que con 10 semillas la ventaja con $c=0$ solo se separa de las bandas de error en $\lambda=0.5$, y que al medirse $c$ en ticks enteros el cruce no puede afinarse más sin variar $\kappa$, de modo que el siguiente paso (la observabilidad parcial) pondrá a prueba si monitorear sobre una creencia obsoleta agrava esta desventaja o si es el lazo abierto el que pierde más al no ver el fallo hasta llegar.

## ¿Cuál es el siguiente paso?

El siguiente paso consiste en reintroducir uno a uno los elementos fijados en esta versión, de modo que cada incorporación responda a una hipótesis concreta y su efecto se mida contra el caso base aquí establecido:

1. **Observabilidad parcial ($r$ finito):** Se limita la visión a un radio $r$ (es decir, qué parte del entorno tiene el agente en su contexto), con la hipótesis de que replanificar sobre una creencia obsoleta degrada al replanificador hasta el punto de que comprometerse con el plan resulta mejor.
2. **Oráculo con competencia $p<1$:** Se reemplaza el A* exacto por un oráculo que con probabilidad $1-p$ devuelve una reparación plausible pero equivocada (objetivo obsoleto o ruta con desvío), con el objetivo de encontrar desde qué calidad conviene auto-editarse.
3. **Capacidad $>1$ y reinicio en frío (`COLD_RESTART`):** Se introduce trabajo en vuelo para que abortar tenga costo (`carga_perdida`), lo cual cuantifica cuánto vale reparar el plan en sitio, conservando el trabajo en curso, frente a reiniciar la ejecución desde cero.
4. **Política gobernada (`GOVERNED`):** Se acota la replanificación con un presupuesto $B$ de ediciones, con la hipótesis de que el límite amortigua el *thrashing* medido como `n_ediciones_desperdiciadas` (que es justo lo que sugiere el exceso de replanificaciones del lazo cerrado en este avance), y junto con ello se reintroduce para todas las políticas que replanifican un alcance de edición acotado, es decir una porción del plan (la entrega final) que queda fuera del alcance del agente y que este no puede modificar.
5. **Barrido completo:** Finalmente, se trata de integrar y analizar todos los ejes con 30 réplicas por combinación, sumando `ceguera`, `latencia_deteccion` y el precio de deliberar $\kappa$.

El documento final contendrá la frontera de fase en el plano $(\lambda, c)$ junto con la forma en que $r$, $p$ y $B$ la desplazan, reportada con sus cifras e incertidumbres y traducida en un criterio de cuándo conviene que un agente reescriba su propio plan.